In [ ]:
import pandas as pd
import json
from scipy.stats import mannwhitneyu
import numpy as np
import os

import sys;sys.path.append("./networks")
from utils import resolve_paths

READ_DATA_PATHS, WRITE_DATA_PATHS = resolve_paths(read_datasets=["Graphs Data", "Results Data"], 
                                                write_datasets=["Results Data", "Training Data"])

Loading DATA_PATHS.yaml from C:\Users\ginof\OneDrive\Documents\GitHub\Detecting-Global-Trade-Vulnerabilities-via-Graph-Neural-Networks\data\DATA_PATHS.yaml


In [38]:
ablations = ['Ablation - COI', 'Ablation - ECI', 'Ablation - # Prod', "Ablation - SRCA", "Ablation - Geo-Positional", 'Ablation - HHI', "Ablation - TI", 
             'Ablation - Export Value', 'Ablation - Avg.PCI', 'Ablation - Trade Agreements', 'Ablation - Trustworthiness']
models = ['GCN', "GAT", "SAGE"]
graphs = ["export", "total", "export-layered", "multi-graph-total", "multi-graph-export"]

digits = 2

In [ ]:
baseline = 'No Ablation'

# Results dictionary to store all the results for the ablation study
results = {
    "Graph": [],
    "Method": [],
    "Ablated measure": [],
    "Seed": [],
    "F1 - Positives": []
}

# Read the baseline (No Ablation) results
for model in models:

    # For each graph
    for graph in graphs:

        #baseline_results = []

        # For each seed
        for seed in range(1, 11):

            # Read the seed-specific result for the baseline (No Ablation)
            with open(READ_DATA_PATHS["Results Data"] + f"/{digits}_digits/{baseline}/{model}/{graph}/{seed}/report.json") as f:
                baseline_result = json.load(f)["F1 - Positives"]
                #baseline_results.append(baseline_result)
                
                # Append the baseline result to the results dictionary
                results["Graph"].append(graph)
                results["Method"].append(model)
                results["Ablated measure"].append("No Ablation")
                results["Seed"].append(seed)
                results["F1 - Positives"].append(baseline_result)

# Now read the ablation results and perform the Mann-Whitney U test against the baseline
for ablation in ablations:

    # For each model
    for model in models:

        # For each graph
        for graph in graphs:

            # These ablations are not applicable to certain graph types
            if ((ablation in ["Ablation - Trustworthiness"]) and (graph in ["export-layered", "total", "export"])) or \
            ((ablation in ["Ablation - SRCA", "Ablation - TI"]) and (graph in ["multi-graph-total", "multi-graph-export"])):
                continue

            ablation_results = []
            baseline_results = []

            # For each seed
            for seed in range(1, 11):
                
                # Read the seed-specific result for the ablation
                with open(READ_DATA_PATHS["Results Data"] + f"/{digits}_digits/{ablation}/{model}/{graph}/{seed}/report.json") as f:
                    ablated = json.load(f)["F1 - Positives"]
                    ablation_results.append(ablated)
                
                # Read the seed-specific result for the baseline (No Ablation)
                with open(READ_DATA_PATHS["Results Data"] + f"/{digits}_digits/{baseline}/{model}/{graph}/{seed}/report.json") as f:
                    base = json.load(f)["F1 - Positives"]
                    baseline_results.append(base)

                # Store the results for this seed in the results dictionary
                results["Graph"].append(graph)
                results["Method"].append(model)
                results["Ablated measure"].append(ablation)
                results["Seed"].append(seed)
                results["F1 - Positives"].append(ablated)
                
            # Compute some test statistic and p-value using the Mann-Whitney U test
            stat, pvalue = mannwhitneyu(baseline_results, ablation_results)
            print(
                f"{ablation:<20} | {model:<6} | {graph:<20} | "
                f"Ablation: {np.mean(ablation_results):.4f} | "
                f"Baseline: {np.mean(baseline_results):.4f} | "
                f"Difference: {np.mean(ablation_results) - np.mean(baseline_results):.4f} | "
                f"p-value: {pvalue:.4f} {'(Significant)' if pvalue < (0.05/(len(ablations)*len(models)*len(graphs)-21)) else ''}"
            )
            
    print()

Ablation - COI       | GCN    | export               | Ablation: 0.1115 | Baseline: 0.1058 | Difference: 0.0058 | p-value: 0.0046 
Ablation - COI       | GCN    | total                | Ablation: 0.0966 | Baseline: 0.1168 | Difference: -0.0201 | p-value: 0.0013 
Ablation - COI       | GCN    | export-layered       | Ablation: 0.1025 | Baseline: 0.1025 | Difference: 0.0000 | p-value: 0.6776 
Ablation - COI       | GCN    | multi-graph-total    | Ablation: 0.9030 | Baseline: 0.9220 | Difference: -0.0190 | p-value: 0.0017 
Ablation - COI       | GCN    | multi-graph-export   | Ablation: 0.9109 | Baseline: 0.9165 | Difference: -0.0056 | p-value: 0.0886 
Ablation - COI       | GAT    | export               | Ablation: 0.1282 | Baseline: 0.1239 | Difference: 0.0043 | p-value: 0.9698 
Ablation - COI       | GAT    | total                | Ablation: 0.1264 | Baseline: 0.1307 | Difference: -0.0043 | p-value: 0.5205 
Ablation - COI       | GAT    | export-layered       | Ablation: 0.1019 | Basel

#### Save results as a CSV

In [52]:
results = pd.DataFrame(results)
results.to_csv(WRITE_DATA_PATHS["Results Data"] + f"/{digits}_digits/ablation_results.csv", index=False)